# Generador de Demanda Vehicular Estocastica para SUMO

Basado en datos de la Tabla 14 - Tesis E. Sarango (2025)
Interseccion: Av. Isidro Ayora / Av. 8 de Diciembre - Loja

Red: Simple_Intersection.net.xml
  Entradas: -E5 (Norte->J11), -E4 (Oeste->J11)
  Salidas:  E3 (J11->Este),   E6 (J11->Sur)

Movimientos permitidos (segun connections del .net.xml):
  -E5 -> E6  (Norte->Sur,   recto)
  -E5 -> E3  (Norte->Este,  giro izquierda)
  -E4 -> E3  (Oeste->Este,  recto)
  -E4 -> E6  (Oeste->Sur,   giro derecha)

In [ ]:
import numpy as np
import xml.etree.ElementTree as ET
from xml.dom import minidom

## Parametros Generales

Referencia:
- SEED=42: garantiza reproducibilidad de resultados
- DURACION=3600s (1 hora): periodo de simulacion
- Weibull shape k=2.5: controla asimetria de hora pico
- Weibull scale c=0.55: centra pico en ~55% de duracion
- FLUJO_BASE=1166.9 veh/h: promedio de 7 muestras (Tabla 13 Sarango 2025)

In [ ]:
SEED       = 42
DURACION   = 3600               # segundos
K_WEIBULL  = 2.5                # forma (shape) de Weibull
C_WEIBULL  = 0.55 * DURACION    # escala en segundos, pico ~55%
TOTAL_VEH  = 1167               # flujo promedio 1166.9 veh/h, redondeado

## Rutas por Acceso

Referencia:
- Acceso Oeste (-E4): Av. Isidro Ayora (via principal)
- Acceso Norte (-E5): Av. 8 de Diciembre (via secundaria)
- Probabilidades de giro: 70% recto, 30% giro (estimacion ingenieria de trafico)

In [ ]:
ACCESOS = [
    {
        "nombre": "Oeste",
        "rutas": [
            ("ruta_OE", "-E4", "E3", 0.70),  # Oeste->Este (recto)
            ("ruta_OS", "-E4", "E6", 0.30),  # Oeste->Sur  (giro derecha)
        ]
    },
    {
        "nombre": "Norte",
        "rutas": [
            ("ruta_NS", "-E5", "E6", 0.70),  # Norte->Sur  (recto)
            ("ruta_NE", "-E5", "E3", 0.30),  # Norte->Este (giro izquierda)
        ]
    },
]

## Composicion Vehicular

Referencia:
- Conteo realizado por YOLOv5 en video de campo (Sarango 2025)
- Ligeros: 7032 (69.58%)
- Motos: 885 (8.76%)
- Pesados/Buses: 2189 (21.66%)

In [ ]:
conteo_car  = 7032
conteo_moto = 885
conteo_bus  = 2189

total_tipos = conteo_car + conteo_moto + conteo_bus

TIPOS = [
    ("car",  "passenger",  round(conteo_car  / total_tipos, 4)),
    ("moto", "motorcycle", round(conteo_moto / total_tipos, 4)),
    ("bus",  "bus",        round(conteo_bus  / total_tipos, 4)),
]

for nombre, clase, prop in TIPOS:
    print(f"{nombre:5s} ({clase:10s}): {prop:6.2%}")

car   (passenger ): 69.58%
moto  (motorcycle):  8.76%
bus   (bus       ): 21.66%


## Generacion con Muestreo Directo de Weibull

Referencia:
- Distribucion Weibull modela la forma asimetrica del trafico durante la hora pico
- PDF: f(t) = (k/c) * (t/c)^(k-1) * exp(-(t/c)^k)
- shape k = 2.5: controla la asimetria (pico pronunciado)
- scale c = 0.55*DURACION: centra el pico hacia el 55% de la duracion

Metodo (muestreo directo, sin Poisson):
1. Generar TOTAL_VEH valores uniformes en (0,1)
2. Aplicar transformada inversa de Weibull: T = c * (-ln(1-U))^(1/k)
3. Los tiempos de llegada siguen exactamente una distribucion Weibull

In [ ]:
def generar_tiempos_weibull(seed_offset, total_vehiculos=TOTAL_VEH):
    rng = np.random.default_rng(SEED + seed_offset)
    u = rng.uniform(0, 1, size=total_vehiculos)
    # Inverse transform: T = C_WEIBULL * (-ln(U))^(1/K_WEIBULL)
    tiempos = C_WEIBULL * (-np.log(u)) ** (1 / K_WEIBULL)
    # Truncar a duracion de simulacion
    tiempos = np.clip(tiempos, 0, DURACION)
    return sorted(tiempos.tolist())

## Construccion del XML

Genera el archivo `.rou.xml` compatible con SUMO.

Componentes:
- vTypes: definiciones de vehiculos (car, moto, bus) con modelo Krauss
- routes: rutas permitidas por cada acceso
- vehicles: lista de vehiculos con tiempo de llegada, tipo y ruta

In [ ]:
def construir_xml(vehiculos_por_acceso):
    root = ET.Element("routes")
    root.set("xmlns:xsi", "http://www.w3.org/2001/XMLSchema-instance")
    root.set("xsi:noNamespaceSchemaLocation", "http://sumo.dlr.de/xsd/routes_file.xsd")

    ET.SubElement(root, "vType", id="car",  vClass="passenger",
                  accel="3.0", decel="4.5", sigma="0.5",
                  length="5.0", minGap="2.5", maxSpeed="13.9",
                  carFollowModel="Krauss", color="0.8,0.8,0.8")
    ET.SubElement(root, "vType", id="moto", vClass="motorcycle",
                  accel="3.5", decel="5.0", sigma="0.6",
                  length="2.2", minGap="2.0", maxSpeed="16.7",
                  carFollowModel="Krauss", color="0.9,0.5,0.1")
    ET.SubElement(root, "vType", id="bus",  vClass="bus",
                  accel="2.0", decel="4.5", sigma="0.3",
                  length="12.0", minGap="2.5", maxSpeed="11.1",
                  carFollowModel="Krauss", color="0.2,0.6,0.2")

    for acceso in ACCESOS:
        for rid, e_from, e_to, _ in acceso["rutas"]:
            ET.SubElement(root, "route", id=rid, edges=f"{e_from} {e_to}")

    todos = []
    for acceso, (tiempos, rng_seed) in zip(ACCESOS, vehiculos_por_acceso):
        rng = np.random.default_rng(rng_seed)
        rutas  = acceso["rutas"]
        p_r    = np.array([r[3] for r in rutas]); p_r /= p_r.sum()
        p_t    = np.array([v[2] for v in TIPOS]);  p_t /= p_t.sum()
        idx_r  = rng.choice(len(rutas), size=len(tiempos), p=p_r)
        idx_t  = rng.choice(len(TIPOS), size=len(tiempos), p=p_t)
        for t, ir, it in zip(tiempos, idx_r, idx_t):
            todos.append({"depart": t, "type": TIPOS[it][0], "route": rutas[ir][0]})
    todos.sort(key=lambda x: x["depart"])

    for i, v in enumerate(todos):
        ET.SubElement(root, "vehicle", id=f"veh_{i}", type=v["type"],
                      route=v["route"], depart=f"{v['depart']:.2f}",
                      departLane="best", departSpeed="0")
    return root, len(todos)

def guardar(root, filepath):
    xml_str = ET.tostring(root, encoding='utf-8')
    parsed_str = minidom.parseString(xml_str)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(parsed_str.toprettyxml(indent="    "))

## Generar Archivo de Rutas

Genera `loja_intersection_weibull.rou.xml` con la demanda vehicular.
- 1167 vehiculos por acceso (total 2334 en 1 hora)
- Seed diferente por acceso para independencia estadistica

In [ ]:
tiempos_oeste = generar_tiempos_weibull(seed_offset=0, total_vehiculos=1167)
tiempos_norte = generar_tiempos_weibull(seed_offset=10, total_vehiculos=1167)

vehiculos_por_acceso = [
    (tiempos_oeste, SEED + 1),
    (tiempos_norte, SEED + 11),
]

root, total = construir_xml(vehiculos_por_acceso)
out = r"./loja_intersection_weibull.rou.xml"
guardar(root, out)
print(f"Total de vehiculos inyectados: {total}")
print(f"Archivo listo: {out}")

Total de vehiculos inyectados: 2334
Archivo listo: ./loja_intersection_weibull.rou.xml
